# Modular RAG의 기본 형태
- Query → Rewrite → Retrieve → Rerank → Filter → Compress → Generate → Output
- (최소) Query → Rewrite → Retrieve → Generate

# 실습 예제

## step 1 - import 및 State 정의 

- State는 그래프 전체에서 공유되는 데이터 구조입니다.
- 모든 중간 결과를 하나의 State에 저장하고, 각 Node는 필요한 값만 읽고 씁니다.


In [1]:
from typing import TypedDict, List

class RAGState(TypedDict):
    query           : str        # 사용자 원본 질문 → 처음 입력, 끝까지 보존
    rewritten_query : str        # 재작성된 질문 → Rewrite 노드가 채움
    docs            : List[str]  # 검색된 문서 청크들 → Retrieve 노드가 채움
    answer          : str        # 최종 답변 → Generate 노드가 채움



## Step 2 - Node 3개 정의

### Node 1: Query Rewrite Node 
- 사용자의 질문을 검색에 적합한 형태로 정제

In [2]:
# Node ① Query Rewrite
def rewrite_node(state: RAGState):
    """
    입력: state["query"]
    출력: state["rewritten_query"]
    실제 환경에서는 LLM으로 질문을 정제합니다.
    """
    rewritten = f"{state['query']} (정책 보고서 기준)" # 문자열 추가
    return {"rewritten_query": rewritten}

### 참고 :  실제 LLM을 쓰는 Rewrite 노드와 비교

In [ ]:
# def rewrite_node(state: RAGState):
#     rewritten = llm.invoke(
#         f"다음 질문을 검색에 최적화된 형태로 재작성하세요: {state['query']}"
#     )
#     return {"rewritten_query": rewritten.content}

### Node 2: Retrieve Node 
- 정제된 질문을 기반으로 문서를 검색

In [3]:
# Node ② Retrieve
def retrieve_node(state: RAGState):
    """
    입력: state["rewritten_query"]
    출력: state["docs"]
    실제 환경에서는 Vector DB 검색이 수행됩니다.
    """
    docs = [
        "2025년 농업기술 정책 보고서 요약본",
        "2024년 농업 R&D 투자 방향",
        "농업기술 혁신 전략 백서"
    ]
    return {"docs": docs}

### Node 3: Generate Node
- 검색된 문서를 바탕으로 최종 답변을 생성

In [4]:
# Node ③ Generate
def generate_node(state: RAGState):
    """
    입력: state["docs"]
    출력: state["answer"]
    실제 환경에서는 LLM 호출이 이루어집니다.
    """
    docs   = state["docs"]
    answer = f"총 {len(docs)}개의 문서를 기반으로 2025년 농업기술 정책을 요약했습니다."
    return {"answer": answer}

## Step 3 - Graph 구성 (Node와 Edge 연결)

- 정의한 Node들을 그래프로 연결
- 실행 순서를 코드로 직접 호출하지 않고, Node 간의 흐름 구조를 선언적으로 정의
- StateGraph : 노드 간에 데이터를 주고받는 공유 메모리

In [5]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(RAGState)
graph.add_node("rewrite", rewrite_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
# graph.add_conditional_edges("retrieve", should_compress, {
#     "compress": "compress",
#     "generate": "generate"
# })

graph.add_edge(START,      "rewrite")
graph.add_edge("rewrite",  "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()

# 스트림 방식 실행
initial_state = {"query": "2025년 농업기술 관련 정책 보고서를 요약해줘"}
for event in app.stream(initial_state):
    print(event)

{'rewrite': {'rewritten_query': '2025년 농업기술 관련 정책 보고서를 요약해줘 (정책 보고서 기준)'}}
{'retrieve': {'docs': ['2025년 농업기술 정책 보고서 요약본', '2024년 농업 R&D 투자 방향', '농업기술 혁신 전략 백서']}}
{'generate': {'answer': '총 3개의 문서를 기반으로 2025년 농업기술 정책을 요약했습니다.'}}


### invoke 방식: 최종 결과만 반환

In [6]:
initial_state = {
    "query": "2025년 농업기술 관련 정책 보고서를 요약해줘"
}

result = app.invoke(initial_state)
print(result["answer"])

총 3개의 문서를 기반으로 2025년 농업기술 정책을 요약했습니다.


## Step 4 - 조건 기반 Routing (State를 이용한 실행 분기)

Query → Rewrite → Retrieve                   
. . . . . . . . . . . . . . . . . . . .  ├─ (docs > 2) → Compress → Generate                 
. . . . . . . . . . . . . . . . . . . .  └─ (docs ≤ 2) → Generate              

### Step 4.1 — Compress Node 추가

In [7]:
def compress_node (state: RAGState):
    """
    입력: state["docs"]
    출력: state["docs"] (압축된 버전)
    문서 수를 줄이거나 요약하는 단계.
    실제 환경에서는 LLM 요약 또는 chunk 축약이 수행됩니다.
    """
    docs            = state["docs"]
    compressed_docs = docs[:2]      # 앞 2개만 사용
    return {"docs": compressed_docs}

### Step 4.2 — 조건 함수 (Router) 정의

In [8]:
def should_compress(state: RAGState) -> str:
    """
    docs 개수가 많으면 Compress Node로 이동
    그렇지 않으면 바로 Generate Node로 이동
    """
    if len(state["docs"]) > 2:
        return "compress"
    return "generate"

### Step 4.3 — 조건 분기 Graph 조립

In [9]:
from langgraph.graph import StateGraph, END, START

graph = StateGraph(RAGState)

graph.add_node("rewrite",  rewrite_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("compress", compress_node)
graph.add_node("generate", generate_node)

graph.add_edge(START,     "rewrite")
graph.add_edge("rewrite", "retrieve")

# 조건부 엣지 — retrieve 이후 should_compress 함수로 분기 결정
graph.add_conditional_edges(
    "retrieve",       # 출발 노드
    should_compress,  # 판단 함수
    {
        "compress": "compress",   # "compress" 반환 시 → compress 노드
        "generate": "generate"    # "generate" 반환 시 → generate 노드
    }
)

graph.add_edge("compress", "generate")
graph.add_edge("generate", END)

app = graph.compile()

### Step 4.4 — 실행

In [10]:
initial_state = {"query": "2025년 농업기술 관련 정책 보고서를 요약해줘"}

for event in app.stream(initial_state):
    print(event)

{'rewrite': {'rewritten_query': '2025년 농업기술 관련 정책 보고서를 요약해줘 (정책 보고서 기준)'}}
{'retrieve': {'docs': ['2025년 농업기술 정책 보고서 요약본', '2024년 농업 R&D 투자 방향', '농업기술 혁신 전략 백서']}}
{'compress': {'docs': ['2025년 농업기술 정책 보고서 요약본', '2024년 농업 R&D 투자 방향']}}
{'generate': {'answer': '총 2개의 문서를 기반으로 2025년 농업기술 정책을 요약했습니다.'}}
